In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:
df = pd.read_csv("../Data/Clean_Flight_Data.csv")

df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Journey_Month,Journey_Day,Departure_Hour
0,IndiGo,2019-03-24,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,March,Sunday,22
1,Air India,2019-05-01,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,May,Wednesday,5
2,Jet Airways,2019-06-09,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,June,Sunday,9
3,IndiGo,2019-05-12,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,May,Sunday,18
4,IndiGo,2019-03-01,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,March,Friday,16


In [3]:
df.columns

Index(['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Route',
       'Dep_Time', 'Arrival_Time', 'Duration', 'Total_Stops',
       'Additional_Info', 'Price', 'Journey_Month', 'Journey_Day',
       'Departure_Hour'],
      dtype='object')

In [5]:
data = pd.read_csv("../Data/Clean_Flight_Data.csv")

data.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Journey_Month,Journey_Day,Departure_Hour
0,IndiGo,2019-03-24,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897,March,Sunday,22
1,Air India,2019-05-01,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662,May,Wednesday,5
2,Jet Airways,2019-06-09,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882,June,Sunday,9
3,IndiGo,2019-05-12,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218,May,Sunday,18
4,IndiGo,2019-03-01,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302,March,Friday,16


In [6]:
data.dtypes

Airline            object
Date_of_Journey    object
Source             object
Destination        object
Route              object
Dep_Time           object
Arrival_Time       object
Duration           object
Total_Stops        object
Additional_Info    object
Price               int64
Journey_Month      object
Journey_Day        object
Departure_Hour      int64
dtype: object

In [34]:
data = df.copy()

# Convert journey date
data["Date_of_Journey"] = pd.to_datetime(data["Date_of_Journey"])

# Extract date features
data["Journey_Day"] = data["Date_of_Journey"].dt.day
data["Journey_Month"] = data["Date_of_Journey"].dt.month

# Departure time
data["Dep_Time"] = pd.to_datetime(data["Dep_Time"], format="%H:%M")

data["Departure_Hour"] = data["Dep_Time"].dt.hour
data["Departure_Minute"] = data["Dep_Time"].dt.minute

data.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price,Journey_Month,Journey_Day,Departure_Hour,Departure_Minute
0,IndiGo,2019-03-24,Banglore,New Delhi,BLR → DEL,1900-01-01 22:20:00,01:10 22 Mar,2h 50m,non-stop,No info,3897,3,24,22,20
1,Air India,2019-05-01,Kolkata,Banglore,CCU → IXR → BBI → BLR,1900-01-01 05:50:00,13:15,7h 25m,2 stops,No info,7662,5,1,5,50
2,Jet Airways,2019-06-09,Delhi,Cochin,DEL → LKO → BOM → COK,1900-01-01 09:25:00,04:25 10 Jun,19h,2 stops,No info,13882,6,9,9,25
3,IndiGo,2019-05-12,Kolkata,Banglore,CCU → NAG → BLR,1900-01-01 18:05:00,23:30,5h 25m,1 stop,No info,6218,5,12,18,5
4,IndiGo,2019-03-01,Banglore,New Delhi,BLR → NAG → DEL,1900-01-01 16:50:00,21:35,4h 45m,1 stop,No info,13302,3,1,16,50


In [35]:
# Keep only the time part (HH:MM)
data["Arrival_Time"] = data["Arrival_Time"].str[:5]

# Convert to datetime
data["Arrival_Time"] = pd.to_datetime(
    data["Arrival_Time"],
    format="%H:%M"
)

# Extract hour and minute
data["Arrival_Hour"] = data["Arrival_Time"].dt.hour
data["Arrival_Minute"] = data["Arrival_Time"].dt.minute

In [36]:
def convert_duration(duration):
    hours = 0
    minutes = 0

    if "h" in duration:
        hours = int(duration.split("h")[0])

    if "m" in duration:
        minutes = int(duration.split("m")[0].split()[-1])

    return hours * 60 + minutes


data["Duration_Minutes"] = data["Duration"].apply(convert_duration)

In [37]:
data.drop(
    columns=[
        "Date_of_Journey",
        "Dep_Time",
        "Arrival_Time",
        "Duration",
        "Route"
    ],
    inplace=True
)

In [38]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10463 entries, 0 to 10462
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Airline           10463 non-null  object
 1   Source            10463 non-null  object
 2   Destination       10463 non-null  object
 3   Total_Stops       10463 non-null  object
 4   Additional_Info   10463 non-null  object
 5   Price             10463 non-null  int64 
 6   Journey_Month     10463 non-null  int32 
 7   Journey_Day       10463 non-null  int32 
 8   Departure_Hour    10463 non-null  int32 
 9   Departure_Minute  10463 non-null  int32 
 10  Arrival_Hour      10463 non-null  int32 
 11  Arrival_Minute    10463 non-null  int32 
 12  Duration_Minutes  10463 non-null  int64 
dtypes: int32(6), int64(2), object(5)
memory usage: 817.6+ KB


In [39]:
# Target
y = data["Price"]

# Features
X = data.drop("Price", axis=1)

X.head()

,Airline,Source,Destination,Total_Stops,Additional_Info,Journey_Month,Journey_Day,Departure_Hour,Departure_Minute,Arrival_Hour,Arrival_Minute,Duration_Minutes
0,IndiGo,Banglore,New Delhi,non-stop,No info,3,24,22,20,1,10,170
1,Air India,Kolkata,Banglore,2 stops,No info,5,1,5,50,13,15,445
2,Jet Airways,Delhi,Cochin,2 stops,No info,6,9,9,25,4,25,1140
3,IndiGo,Kolkata,Banglore,1 stop,No info,5,12,18,5,23,30,325
4,IndiGo,Banglore,New Delhi,1 stop,No info,3,1,16,50,21,35,285


In [40]:
categorical_features = [
    "Airline",
    "Source",
    "Destination",
    "Total_Stops",
    "Additional_Info"
]

numerical_features = [
    "Journey_Day",
    "Journey_Month",
    "Departure_Hour",
    "Departure_Minute",
    "Arrival_Hour",
    "Arrival_Minute",
    "Duration_Minutes"
]

In [41]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

In [42]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [44]:
data.isnull().sum()

Airline             0
Source              0
Destination         0
Total_Stops         0
Additional_Info     0
Price               0
Journey_Month       0
Journey_Day         0
Departure_Hour      0
Departure_Minute    0
Arrival_Hour        0
Arrival_Minute      0
Duration_Minutes    0
dtype: int64

In [45]:
data[data.isnull().any(axis=1)].head()

,Airline,Source,Destination,Total_Stops,Additional_Info,Price,Journey_Month,Journey_Day,Departure_Hour,Departure_Minute,Arrival_Hour,Arrival_Minute,Duration_Minutes


In [46]:
df["Arrival_Time"].head(10)

0    01:10 22 Mar
1           13:15
2    04:25 10 Jun
3           23:30
4           21:35
5           11:25
6    10:25 13 Mar
7    05:05 02 Mar
8    10:25 13 Mar
9           19:15
Name: Arrival_Time, dtype: object

In [47]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](12,)","['Airline','Source','Destination',...,'Arrival_Hour','Arrival_Minute', 'Duration_Minutes']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,12
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...), ('num', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. Thi

In [48]:
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

# Predictions
y_pred = pipeline.predict(X_test)

# Metrics
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

print(f"R² Score : {r2:.4f}")
print(f"MAE      : ₹{mae:.2f}")
print(f"RMSE     : ₹{rmse:.2f}")

R² Score : 0.8943
MAE      : ₹658.92
RMSE     : ₹1484.97


In [49]:
import joblib

joblib.dump(pipeline, "../Models/flight_price_pipeline.pkl")

print("Pipeline saved successfully!")

Pipeline saved successfully!
